# Laboratorio 3. Seleccion de arquitecturas y plan de procesamiento

Fabian Prado Dluzniewski 23427

Abby Donis 22440

Hansel Lopez 19026

Este notebook define las arquitecturas que se van a entrenar en la entrega
final y el plan de procesamiento de imagenes. Corresponde a la parte del avance
que pide seleccion de modelos y plan, por lo que aqui se justifican y se
verifican las arquitecturas, no se hace el tuneo completo.

Requiere haber corrido `01_EDA.ipynb` y `02_Preprocesamiento.ipynb`, que dejan
los tensores en `data/procesado/`.

### IMPORTS

In [1]:
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

DISPOSITIVO = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"dispositivo: {DISPOSITIVO}")

dispositivo: mps


### CARGA DE LOS DATOS PREPROCESADOS

In [2]:
from pathlib import Path

DIR_PROC = Path("./data/procesado")
norma = np.load(DIR_PROC / "normalizacion.npz", allow_pickle=True)
MEDIA, DESV = norma["media"], norma["desv"]
CLASES = list(norma["clases"])
LADO = int(norma["lado"])
N_CLASES = len(CLASES)


def cargar(grupo):
    d = np.load(DIR_PROC / f"{grupo}.npz")
    X = torch.from_numpy(d["X"]).permute(0, 3, 1, 2).float().div_(255.0)
    X = (X - torch.tensor(MEDIA).view(3, 1, 1)) / torch.tensor(DESV).view(3, 1, 1)
    return TensorDataset(X, torch.from_numpy(d["y"]))


conjuntos = {g: cargar(g) for g in ["train", "val", "test"]}
for g, ds in conjuntos.items():
    print(f"{g:6} {len(ds):,} imagenes")
print(f"clases {N_CLASES}   entrada {3}x{LADO}x{LADO}")

train  12,180 imagenes
val    2,610 imagenes
test   2,610 imagenes
clases 29   entrada 3x64x64


## 4.1. Criterios de seleccion

Las arquitecturas se eligen a partir de lo que mostro el analisis exploratorio,
no de una lista generica.

El dataset esta balanceado y tiene fondo constante, asi que el problema no es
de datos escasos ni de clases raras. La dificultad esta en distinguir clases
que comparten silueta, sobre todo el bloque U, V y R, donde la diferencia son
unos pocos pixeles entre dedos. Eso pide filtros que capten detalle local fino,
es decir convoluciones pequenas de 3x3 y suficiente profundidad para componer
bordes en formas.

Al mismo tiempo, la redundancia entre cuadros consecutivos implica que el
numero efectivo de ejemplos distintos es mucho menor que 12,180. Un modelo con
demasiados parametros memoriza la sesion de grabacion en lugar de aprender la
mano, asi que la regularizacion importa mas que el tamano.

Se comparan cuatro familias.

## 4.2. Arquitecturas de red convolucional

**CNN A, linea base.** Tres bloques convolucionales con 32, 64 y 128 filtros,
cada uno seguido de agrupamiento maximo. Es la arquitectura minima capaz de
llegar a un campo receptivo que cubra la mano completa. Sirve de referencia
para medir si la complejidad adicional de la segunda red se justifica.

**CNN B, profunda y regularizada.** Cuatro bloques con normalizacion por lotes
y descarte, y agrupamiento promedio global en lugar de aplanar. La
normalizacion por lotes estabiliza el entrenamiento y permite tasas de
aprendizaje mayores. El agrupamiento promedio global reduce muchisimo los
parametros de la capa final, que es justo donde una red de este tipo tiende a
memorizar.

In [3]:
class CNNBase(nn.Module):
    def __init__(self, n_clases=N_CLASES, filtros=(32, 64, 128), p_descarte=0.25):
        super().__init__()
        capas, entrada = [], 3
        for f in filtros:
            capas += [nn.Conv2d(entrada, f, 3, padding=1), nn.ReLU(),
                      nn.MaxPool2d(2)]
            entrada = f
        self.rasgos = nn.Sequential(*capas)
        lado_final = LADO // (2 ** len(filtros))
        self.clasificador = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p_descarte),
            nn.Linear(entrada * lado_final * lado_final, 256), nn.ReLU(),
            nn.Dropout(p_descarte),
            nn.Linear(256, n_clases),
        )

    def forward(self, x):
        return self.clasificador(self.rasgos(x))


class CNNProfunda(nn.Module):
    def __init__(self, n_clases=N_CLASES, filtros=(32, 64, 128, 256), p_descarte=0.3):
        super().__init__()
        capas, entrada = [], 3
        for f in filtros:
            capas += [nn.Conv2d(entrada, f, 3, padding=1), nn.BatchNorm2d(f), nn.ReLU(),
                      nn.Conv2d(f, f, 3, padding=1), nn.BatchNorm2d(f), nn.ReLU(),
                      nn.MaxPool2d(2), nn.Dropout2d(p_descarte / 2)]
            entrada = f
        self.rasgos = nn.Sequential(*capas)
        self.clasificador = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(p_descarte),
            nn.Linear(entrada, n_clases),
        )

    def forward(self, x):
        return self.clasificador(self.rasgos(x))

## 4.3. Red densa simple

Corresponde al ejercicio 5. Aplana la imagen a un vector de 12,288 valores y la
pasa por dos capas ocultas. Se incluye para cuantificar cuanto aporta la
convolucion, porque al aplanar se pierde toda la relacion espacial entre
pixeles vecinos, que es precisamente lo que distingue una mano de otra.

In [4]:
class RedDensa(nn.Module):
    def __init__(self, n_clases=N_CLASES, ocultas=(512, 256), p_descarte=0.3):
        super().__init__()
        capas, entrada = [nn.Flatten()], 3 * LADO * LADO
        for h in ocultas:
            capas += [nn.Linear(entrada, h), nn.ReLU(), nn.Dropout(p_descarte)]
            entrada = h
        capas.append(nn.Linear(entrada, n_clases))
        self.red = nn.Sequential(*capas)

    def forward(self, x):
        return self.red(x)

## 4.4. Comparacion de tamano y verificacion

Se confirma que las tres redes producen la forma de salida correcta y se
compara su numero de parametros.

In [5]:
def n_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)


lote_prueba = torch.randn(8, 3, LADO, LADO)
filas = []
for nombre, modelo in [("CNN A base", CNNBase()),
                       ("CNN B profunda", CNNProfunda()),
                       ("Red densa", RedDensa())]:
    with torch.no_grad():
        salida = modelo(lote_prueba)
    filas.append({
        "Modelo": nombre,
        "Parametros": n_parametros(modelo),
        "Salida": str(tuple(salida.shape)),
    })

tabla = pd.DataFrame(filas)
display(tabla.style.format({"Parametros": "{:,}"}))
print(f"todas las salidas deben ser (8, {N_CLASES})")

,Modelo,Parametros,Salida
0,CNN A base,"2,198,109","(8, 29)"
1,CNN B profunda,"1,181,629","(8, 29)"
2,Red densa,"6,430,749","(8, 29)"


todas las salidas deben ser (8, 29)


La red densa tiene mas parametros que las dos redes convolucionales juntas y
aun asi no puede ver que dos pixeles vecinos estan relacionados. Es el
argumento cuantitativo de por que se espera que pierda contra las CNN, y es lo
que el ejercicio 5 pide discutir.